In [1]:
import itertools
import multiprocessing
import os
import pathlib

import pandas as pd
import tifffile
import tomli
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
if in_notebook:
    import tqdm.auto as tqdm
else:
    import tqdm
bandicoot_mount_path = pathlib.Path(os.path.expanduser("~/mnt/bandicoot"))
bandicoot_mount_path = bandicoot_check(bandicoot_mount_path, root_dir)

In [2]:
patient_id_file = pathlib.Path(f"{bandicoot_mount_path}/data/patient_IDs.txt").resolve(
    strict=True
)
patients = pd.read_csv(
    patient_id_file, header=None, names=["patient_id"]
).patient_id.tolist()

load_combinations_path = pathlib.Path(
    f"{root_dir}/3.cellprofiling/load_data/load_combinations.txt"
)
load_combinations_path.parent.mkdir(parents=True, exist_ok=True)

channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

channels = ["DNA", "ER", "Mito", "AGP"]
compartments = ["Organoid", "Nuclei", "Cytoplasm", "Cell"]

In [3]:
patients = patients[:4]

In [4]:
list_of_dicts = []

for patient in tqdm.tqdm(patients, desc="Patients", unit="patient", leave=True):
    final_dict = {
        "patient": [],
        "well_fov": [],
        "image_path": [],
        "image_shape": [],
    }
    patient_well_fovs = sorted(
        [
            path.name
            for path in (
                bandicoot_mount_path / "data" / patient / "zstack_images"
            ).glob("*")
            if path.is_dir()
        ]
    )
    for well_fov in tqdm.tqdm(
        patient_well_fovs, desc="Well/FOV", unit="well_fov", leave=False
    ):
        images = sorted(
            (bandicoot_mount_path / "data" / patient / "zstack_images" / well_fov).glob(
                "*.tif*"
            )
        )
        masks = sorted(
            (
                bandicoot_mount_path
                / "data"
                / patient
                / "segmentation_masks"
                / well_fov
            ).glob("*.tif*")
        )
        for image in images:
            final_dict["patient"].append(patient)
            final_dict["well_fov"].append(well_fov)
            final_dict["image_path"].append(image)
            final_dict["image_shape"].append(tifffile.TiffFile(image).series[0].shape)
        for mask in masks:
            final_dict["patient"].append(patient)
            final_dict["well_fov"].append(well_fov)
            final_dict["image_path"].append(mask)
            final_dict["image_shape"].append(tifffile.TiffFile(mask).series[0].shape)
    list_of_dicts.append(final_dict)

df = pd.DataFrame(
    {
        "patient": [],
        "well_fov": [],
        "image_path": [],
        "image_shape": [],
    }
)
for d in list_of_dicts:
    df = [pd.concat([df, pd.DataFrame(d)], ignore_index=True) for d in list_of_dicts]

Patients:   0%|          | 0/4 [00:00<?, ?patient/s]

Well/FOV:   0%|          | 0/104 [00:00<?, ?well_fov/s]

Well/FOV:   0%|          | 0/350 [00:00<?, ?well_fov/s]

Well/FOV:   0%|          | 0/122 [00:00<?, ?well_fov/s]

KeyboardInterrupt: 

In [ ]:
mismatched_shapes = 0
mismatched_shapes_list = []

# verify shapes for each patient/well_fov combination are the same
for (patient, well_fov), group in df.groupby(["patient", "well_fov"]):
    image_shapes = group.image_shape.unique()
    if len(image_shapes) > 1:
        mismatched_shapes += 1
        mismatched_shapes_list.append((patient, well_fov))
print(
    f"Number of patient/well_fov combinations with mismatched shapes: {mismatched_shapes}"
)

In [ ]:
df.groupby(["patient"]).well_fov.nunique()